In [2]:
# https://chatgpt.com/c/68bc2472-baec-8331-88cd-1410188ed832

# Conda activate ITP259, zive_dask.

# Patobulintas variantas su išsamesne dokumentacija ir klaidų tvarkymu.

"""
EKG triukšmų analizė su U-Net (Keras) modeliu.
- Patikrina modelio/žurnalų suderinamumą.
- Užkrauna modelį ir įrašų sąrašą iš Excel (ignoruoja tag == 9999).
- Filtruoja signalą, randa outliers/rdropouts, U-Net triukšmus.
- Skaičiuoja metrikas ir išsaugo .txt + .xlsx + summary.txt.

Reikalingi moduliai iš TRIUKSMU_DETEKTAVIMAS:
- zive_util_ml.get_ecg_signal
- use_ecg_denoising_util.{bandpass_filter, find_outliers_rdropouts, merge_lists_of_tuples, run_ecg_denoising_pipeline}
"""

from __future__ import annotations

import json
import logging
from pathlib import Path
from typing import List, Optional, Sequence, Tuple

import keras  
import numpy as np
import pandas as pd
import neurokit2 as nk  

import sys

# === Išoriniai moduliai (lygiagretus aplankas) =================================
BASE_PATH = Path().resolve().parent / "TRIUKSMU_DETEKTAVIMAS"
sys.path.append(str(BASE_PATH))

from zive_util_ml import get_ecg_signal  # noqa: E402

from use_ecg_denoising_util import (  # noqa: E402
    find_outliers_rdropouts,
    merge_lists_of_tuples,
    run_ecg_denoising_pipeline,
)

# highpass filter
def highpass_filter(signal, fs=200, lowcut=0.5, highcut=None, method="butterworth", order=5):
    filtered = nk.signal_filter(signal, fs, lowcut, highcut, method, order)
    return np.asarray(filtered, dtype=float)


# === Konfigūracija ==============================================================
FS = 200
SEGMENT_LENGTH = 1024
OVERLAP = 0.5
CONFIG = {"FS": FS, "SEGMENT_LENGTH": SEGMENT_LENGTH, "OVERLAP": OVERLAP}

THRESHOLD = 0.08  # U-Net residual noise threshold

# PROJEKTO_APLANKAS = Path.home() / "DI/2025_ZIVEO/S-ITP-25-9"
# MODELIO_APLANKAS = PROJEKTO_APLANKAS / "MODEL_UNET"
# LIST_DIR = PROJEKTO_APLANKAS / "DATA"/ "ZIVE_DATA"
# REC_DIR = PROJEKTO_APLANKAS / "DATA"/ "ZIVE_DATA"/ "ecg_npy_all"

# MODEL_FILE_NAME = "resunet_ecg_1024_0_5_3_7.keras"
# MARKER = MODEL_FILE_NAME.removeprefix("resunet_ecg").removesuffix(".keras")  # -> "_1024_0_5_3_7"
# NOISE_LOG_BASENAME = f"noise_log{MARKER}"

# # EXCEL_NAME = "visi_zive_irasai_test.xlsx"
# EXCEL_NAME = "visi_zive_irasai.xlsx"

PROJEKTO_APLANKAS = Path.home() / "DI/2025_ZIVEO/PROJECT_TRAIN_UNET"
MODELIO_APLANKAS = PROJEKTO_APLANKAS / "TEST_UNET/MODEL_UNET"
LIST_DIR = PROJEKTO_APLANKAS / "DATA_ORIG"/ "ecg_zive_npy"
REC_DIR = PROJEKTO_APLANKAS / "DATA_ORIG"/ "ecg_zive_npy"

MODEL_FILE_NAME = "resunet_ecg_1024_0_5_3_7.keras"
MARKER = MODEL_FILE_NAME.removeprefix("resunet_ecg").removesuffix(".keras")  # -> "_1024_0_5_3_7"
NOISE_LOG_BASENAME = f"noise_log{MARKER}"

# EXCEL_NAME = "visi_zive_irasai_test.xlsx"
EXCEL_NAME = "visi_zive_irasai.xlsx"


# === Pagalbinės funkcijos =======================================================


def setup_logging() -> None:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%H:%M:%S",
    )


def check_consistency(model_file_name: str, noise_log_basename: str, config: dict) -> None:
    """Patikrina failų sufiksų ir parametrų suderinamumą."""
    model_suffix = model_file_name.split("resunet_ecg")[-1].replace(".keras", "")
    noise_suffix = noise_log_basename.split("noise_log")[-1]

    if model_suffix != noise_suffix:
        raise ValueError(f"Suffix mismatch:\n  model: {model_suffix}\n  noise: {noise_suffix}")

    seg_str = str(config["SEGMENT_LENGTH"])
    if seg_str not in model_suffix:
        raise ValueError(f"SEGMENT_LENGTH mismatch: expected '{seg_str}' in '{model_suffix}'")

    overlap_str = str(config["OVERLAP"]).replace(".", "_")
    if overlap_str not in model_suffix:
        raise ValueError(f"OVERLAP mismatch: expected '{overlap_str}' in '{model_suffix}'")

    logging.info("Suderinamumas OK | sufiksas:%s | SEGMENT_LENGTH:%s | OVERLAP:%s",
                 model_suffix, seg_str, overlap_str)


def load_model_checked(model_path: Path, segment_len: int) -> keras.Model:
    """Užkrauna Keras modelį ir patikrina input shape == (segment_len, 1)."""
    logging.info("Kraunamas modelis: %s", model_path)
    try:
        model = keras.models.load_model(str(model_path))
    except Exception as exc:  # noqa: BLE001
        raise ValueError(f"Failed to load model from {model_path}: {exc}") from exc

    if not hasattr(model, "input_shape") or model.input_shape is None:
        raise ValueError("Loaded model has no valid input_shape")

    expected = (segment_len, 1)
    if model.input_shape[1:] != expected:
        raise ValueError(f"Model expects {model.input_shape[1:]}, script uses {expected}")

    logging.info("Modelis OK, input_shape: %s", model.input_shape)
    return model


def get_ecg_noise_indices_annotated_ext(json_path: Path) -> Optional[List[Tuple[int, int]]]:
    """
    Grąžina [(startIndex, endIndex), ...] iš JSON 'noises_annotated'.
    Jei failo nėra ar raktas neegzistuoja – grąžina None (aiškus signalas).
    """
    if not json_path.exists():
        return None

    with open(json_path, "r", encoding="utf-8", errors="ignore") as f:
        data = json.load(f)

    items = data.get("noises_annotated")
    if not isinstance(items, list):
        return None

    out: List[Tuple[int, int]] = []
    for it in items:
        try:
            out.append((int(it["startIndex"]), int(it["endIndex"])))
        except (KeyError, TypeError, ValueError):
            # praleidžiam brokuotą įrašą, bet netrukdom kitų
            continue
    return out if out else None


def safe_int(value: object) -> Optional[int]:
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def interval_coverage_percent(intervals: Sequence[Tuple[int, int]], total_len: int) -> float:
    """Padengimo dalis % (indeksai laikomi įtrauktiniais: +1)."""
    if total_len <= 0:
        return 0.0
    covered = sum(max(0, end - start + 1) for start, end in intervals)
    return (covered / total_len) * 100.0


def fmt_pct(x: Optional[float | int]) -> str:
    return f"{x:.1f}%" if isinstance(x, (float, int)) else "-"


def read_filenames_from_excel(xlsx_path: Path) -> tuple[list[str], pd.DataFrame]:
    """Skaito Excel, grąžina sąrašą `.npy` failų ir visą DF (meta duomenims paimti)."""
    logging.info("Skaitomas Excel: %s", xlsx_path)
    df = pd.read_excel(xlsx_path, dtype=str)  # saugu tolimesnėms konversijoms
    if "filename" not in df.columns or "tag" not in df.columns:
        raise ValueError("Excel must contain columns: 'filename' and 'tag'")

    filtered = df[df["tag"] != "9999"]
    names = []
    for s in filtered["filename"].dropna():
        name = str(s).strip()
        if not name.endswith(".npy"):
            name += ".npy"
        names.append(name)

    logging.info("Atrinkta įrašų: %d", len(names))
    return names, df


# === Pagrindinis srautas ========================================================


def main() -> None:
    setup_logging()

    print("\nĮRAŠŲ KOKYBĖS ĮVERTINIMAS NUSTATANT TRIUKŠMŲ FRAGMENTŲ KIEKĮ ĮRAŠUOSE")
    print("Surandamos išskirtys (outliers), rspragos (rdropouts) ir U-Net tipo triukšmai\n")
    logging.info("CONFIG=%s | THRESHOLD=%.3f", CONFIG, THRESHOLD)
    logging.info("Projekto aplankas: %s | Modelio aplankas: %s", PROJEKTO_APLANKAS, MODELIO_APLANKAS)
    logging.info("Model: %s | Noise log: %s.[txt|xlsx]", MODEL_FILE_NAME, NOISE_LOG_BASENAME)

    check_consistency(MODEL_FILE_NAME, NOISE_LOG_BASENAME, CONFIG)

    model = load_model_checked(MODELIO_APLANKAS / MODEL_FILE_NAME, SEGMENT_LENGTH)
    file_names, df_meta = read_filenames_from_excel(LIST_DIR / EXCEL_NAME)

    # Kaupiame rezultatus
    text_log: list[str] = []
    rows_xlsx: list[dict] = []

    total_tp_all = 0.0
    n_files = len(file_names)

    n_annotated_files = 0
    sum_atp_across_annotated = 0.0
    sum_tp_in_annotated_files = 0.0  # bendras tp (algoritmo) tik tuose įrašuose, kurie turi anotacijas

    for i, fname in enumerate(file_names, start=1):
        fpath = REC_DIR / fname

        try:
            ecg = get_ecg_signal(str(fpath))
            ann = get_ecg_noise_indices_annotated_ext(fpath.with_suffix(".json"))
        except Exception as exc:  # noqa: BLE001
            raise ValueError(f"Failed to load {fname}: {exc}") from exc

        ecg_f = highpass_filter(ecg)

        # (A) Anotuotų triukšmų padengimas
        if ann is not None:
            n_annotated_files += 1
            merged_ann = merge_lists_of_tuples(ann)
            atp = interval_coverage_percent(merged_ann, len(ecg_f))
            sum_atp_across_annotated += atp
            line3 = f"Anotuotų triukšmų kiekis: {len(ann)} | Anotuotų triukšmų dalis: {atp:.2f}%"
        else:
            atp = None
            line3 = "Anotuotų triukšmų nėra"

        # (B) Outliers + rdropouts + U-Net
        ecg_clean, out_idx, rdr_idx = find_outliers_rdropouts(ecg_f)
        _, noi_idx = run_ecg_denoising_pipeline(ecg_clean, model, CONFIG, THRESHOLD)

        merged_all = merge_lists_of_tuples([*noi_idx, *rdr_idx, *out_idx])
        tp = interval_coverage_percent(merged_all, len(ecg_f))
        total_tp_all += tp
        if ann is not None:
            sum_tp_in_annotated_files += tp

        # Meta duomenys iš Excel pagal stem
        stem = fpath.stem
        row = df_meta[df_meta["filename"] == stem].head(1)

        if not row.empty:
            quality = safe_int(row.iloc[0].get("quality"))
            noni = row.iloc[0].get("noni")
            tag = safe_int(row.iloc[0].get("tag"))
            mark = row.iloc[0].get("mark")
            N = row.iloc[0].get("N")
            S = row.iloc[0].get("S")
            V = row.iloc[0].get("V")
            comment = row.iloc[0].get("comment")
            line2 = f"quality:{quality} noni:{noni} tag:{tag} mark:{mark} N:{N} S:{S} V:{V} comment:{comment}"
        else:
            quality = tag = None
            noni = mark = N = S = V = comment = None
            line2 = f"Filename {fname} not found."

        line1 = (
            f"\n{i}. {fname} | outliers:{len(out_idx)} rdropouts:{len(rdr_idx)} "
            f"kiti triukšmai:{len(noi_idx)} | triukšmų dalis: {tp:.2f}%"
        )
        print(line1)
        print(line2)
        if ann is not None:
            print(line3)

        text_log.extend([line1, line2])
        if ann is not None:
            text_log.append(line3)

        rows_xlsx.append(
            {
                "fn": fname,
                "qlt": quality,
                "tag": tag,
                "out": len(out_idx),
                "rdr": len(rdr_idx),
                "noi": len(noi_idx),
                "tp": round(tp, 1),
                "anoi": len(ann) if ann is not None else None,
                "atp": round(atp, 1) if atp is not None else None,
            }
        )

    # Išsaugojimas: TXT + XLSX
    MODELIO_APLANKAS.mkdir(parents=True, exist_ok=True)

    noise_txt = MODELIO_APLANKAS / f"{NOISE_LOG_BASENAME}.txt"
    with open(noise_txt, "w", encoding="utf-8") as f:
        for line in text_log:
            f.write(line + "\n")
    print(f"\nTxt noise_log written to: {noise_txt}")

    df_log = pd.DataFrame(rows_xlsx)
    noise_xlsx = MODELIO_APLANKAS / f"{NOISE_LOG_BASENAME}.xlsx"
    df_log.to_excel(noise_xlsx, index=False)
    print(f"Excel noise_log written to: {noise_xlsx}")

    # Summary
    summary_lines: list[str] = []
    summary_lines.append("\nVISŲ ĮRAŠŲ REZULTATAI:")
    summary_lines.append(f"Visų analizuotų įrašų skaičius: {n_files}")

    avg_tp_all = (total_tp_all / n_files) if n_files else 0.0
    summary_lines.append(f"Vidutinė triukšmo dalis (tp) įrašuose: {avg_tp_all:.1f}%")

    summary_lines.append("Vidutinė triukšmo dalis 'tp' per 'quality':")
    for q in (0, 1, 2):
        vals = df_log.loc[df_log["qlt"] == q, "tp"]
        avg_q = round(vals.mean(), 1) if not vals.empty else None
        summary_lines.append(f"quality {q}: {fmt_pct(avg_q)}")

    summary_lines.append("\nANOTUOTŲ ĮRAŠŲ REZULTATAI:")
    summary_lines.append(f"Anotuotų įrašų skaičius: {n_annotated_files}")

    avg_atp = (sum_atp_across_annotated / n_annotated_files) if n_annotated_files else 0.0
    avg_tp_in_annot = (
        (sum_tp_in_annotated_files / n_annotated_files) if n_annotated_files else 0.0
    )
    summary_lines.append(f"Vidutinė anotuotų triukšmų dalis 'atp': {avg_atp:.1f}%")
    summary_lines.append(
        "Vidutinė (bendrų) triukšmų dalis 'tp' anotuotuose įrašuose: "
        f"{avg_tp_in_annot:.1f}%"
    )

    summary_path = MODELIO_APLANKAS / f"summary{MARKER}.txt"
    with open(summary_path, "w", encoding="utf-8") as f:
        for line in summary_lines:
            f.write(line + "\n")

    print(f"Noise_summary written to: {summary_path}")
    print("\n=== FINAL SUMMARY ===")
    with open(summary_path, "r", encoding="utf-8") as f:
        print(f.read())


if __name__ == "__main__":
    main()


16:55:39 | INFO | CONFIG={'FS': 200, 'SEGMENT_LENGTH': 1024, 'OVERLAP': 0.5} | THRESHOLD=0.080
16:55:39 | INFO | Projekto aplankas: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET | Modelio aplankas: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/TEST_UNET/MODEL_UNET
16:55:39 | INFO | Model: resunet_ecg_1024_0_5_3_7.keras | Noise log: noise_log_1024_0_5_3_7.[txt|xlsx]
16:55:39 | INFO | Suderinamumas OK | sufiksas:_1024_0_5_3_7 | SEGMENT_LENGTH:1024 | OVERLAP:0_5
16:55:39 | INFO | Kraunamas modelis: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/TEST_UNET/MODEL_UNET/resunet_ecg_1024_0_5_3_7.keras



ĮRAŠŲ KOKYBĖS ĮVERTINIMAS NUSTATANT TRIUKŠMŲ FRAGMENTŲ KIEKĮ ĮRAŠUOSE
Surandamos išskirtys (outliers), rspragos (rdropouts) ir U-Net tipo triukšmai



16:55:40 | INFO | Modelis OK, input_shape: (None, 1024, 1)
16:55:40 | INFO | Skaitomas Excel: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/DATA_ORIG/ecg_zive_npy/visi_zive_irasai.xlsx
16:55:41 | INFO | Atrinkta įrašų: 1085


main_window_size: 1000 (5.0 secs), sliding_window_size: 200 (1.0 secs)

Check if the conditions are met:
OK. 2*sliding_window_size is less than main_window_size
OK. main_window_size is less than all specified values: t_gap_max, extra_interval, t_start_gap_max, t_end_gap_max


KeyboardInterrupt: 